In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%%bash
set -e

cd /content

if [ -d smallnet/.git ]; then
    cd smallnet
    git fetch origin
    git pull --ff-only origin main
else
    git clone https://github.com/SepehrAkbari/smallnet.git
fi

Cloning into 'smallnet'...
Updating files: 100% (1657/1657), done.


In [3]:
%cd /content/smallnet
!git pull --ff-only origin main
!uv sync
!uv run python -m pytest -q

/content/smallnet
From https://github.com/SepehrAkbari/smallnet
 * branch            main       -> FETCH_HEAD
Already up to date.
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 85 packages in 18ms
Prepared 79 packages in 33.52s                                           
Installed 79 packages in 241ms                              
 + asttokens==3.0.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.3.1
 + executing==2.2.1
 + filelock==3.29.0
 + fonttools==4.63.0
 + fsspec==2026.4.0
 + iniconfig==2.3.0
 + ipykernel==7.2.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.8
 + jedi==0.20.0
 + jinja2==3.1.6
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + jupyterlab-widgets==3.0.16
 + kiwisolver==1.5.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + matplotlib-inline==0.2.2
 + mpmath==1.3.0
 + nest-as

In [12]:
!mkdir -p /content/smallnet/model

!rsync -a \
  /content/drive/MyDrive/model/ \
  /content/smallnet/model/

In [13]:
!ls

AGENTS.md  data     model	    README.md  scripts	uv.lock
colab	   docs     notebook	    res        src
configs    LICENSE  pyproject.toml  results    tests


In [14]:
!ls -lh model/best_model.pth
!sha256sum model/best_model.pth

-rw------- 1 root root 513M May 21 16:33 model/best_model.pth
1576b92ebb359a519d57d489f8ee823c033588fbd7c77b1c9091606f42fb74a3  model/best_model.pth


In [6]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage rank512-stability \
  --device cuda \
  --ranks 512 \
  --seeds 0 1 2 \
  --iteration-budgets 200 400 \
  --repetitions 0 1 \
  --optimization-precisions float32

Wrote:
  /content/smallnet/results/camvid_vgg_cp/rank512_stability/rank512_stability_metadata.json


In [7]:
!rsync -a \
  /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a \
  /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/

In [17]:
%cd /content/smallnet

!mkdir -p results/camvid_vgg_cp
!mkdir -p results/paper

!rsync -av \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/ \
  /content/smallnet/results/camvid_vgg_cp/

!rsync -av \
  /content/drive/MyDrive/smallnet_colab_backup/paper/ \
  /content/smallnet/results/paper/

/content/smallnet
sending incremental file list
./
cp_finetune_config_used.json
cp_iteration_sensitivity_budget_transitions.csv
cp_iteration_sensitivity_config_used.json
cp_iteration_sensitivity_metadata.json
cp_iteration_sensitivity_rank_summary.csv
cp_iteration_sensitivity_summary.csv
cp_zero_shot_config_used.json
cp_zero_shot_metadata.json
cp_zero_shot_summary.csv
dataset_class_counts.csv
dataset_mask_forensics.json
dataset_unknown_colors_by_file.csv
dataset_validation_config_used.json
dataset_validation_report.json
dataset_validation_summary.csv
dense_eval_config_used.json
dense_eval_metadata.json
dense_eval_summary.csv
existing_finetuned_config_used.json
existing_finetuned_metadata.json
existing_finetuned_summary.csv
profile_config_used.json
profile_metadata.json
profile_summary.csv
rank_diagnostics_config_used.json
rank_diagnostics_metadata.json
rank_diagnostics_summary.csv
reconstruction_config_used.json
reconstruction_figures_config_used.json
reconstruction_figures_metadata.jso

In [18]:
!find results/camvid_vgg_cp/rank512_stability \
  -maxdepth 1 \
  -type f \
  -print \
  | sort

!ls -lh \
  results/paper/rank512_stability_audit.md \
  results/paper/figures/rank512_stability.csv \
  results/paper/figures/rank512_stability.pdf \
  results/paper/figures/rank512_stability.png

results/camvid_vgg_cp/rank512_stability/rank512_stability_aggregates.csv
results/camvid_vgg_cp/rank512_stability/rank512_stability_config_used.json
results/camvid_vgg_cp/rank512_stability/rank512_stability_metadata.json
results/camvid_vgg_cp/rank512_stability/rank512_stability_summary.csv
-rw------- 1 root root 1.1K Jul 23 15:31 results/paper/figures/rank512_stability.csv
-rw------- 1 root root  21K Jul 23 15:31 results/paper/figures/rank512_stability.pdf
-rw------- 1 root root 153K Jul 23 15:31 results/paper/figures/rank512_stability.png
-rw------- 1 root root 1.1K Jul 23 15:31 results/paper/rank512_stability_audit.md


In [19]:
import pandas as pd
from pathlib import Path

path = Path(
    "results/camvid_vgg_cp/rank512_stability/"
    "rank512_stability_summary.csv"
)

df = pd.read_csv(path)

for column in [
    "rank",
    "seed",
    "iteration_budget",
    "repetition",
]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

lean = df[
    (df["optimization_precision"] == "float32")
    & (df["iteration_budget"].isin([200, 400]))
    & (df["repetition"].isin([0, 1]))
]

display(
    lean[
        [
            "seed",
            "iteration_budget",
            "repetition",
            "actual_relative_squared_frobenius_error",
            "status",
        ]
    ].sort_values(
        ["seed", "repetition", "iteration_budget"]
    )
)

print("Rows:", len(lean))
print("Status counts:")
print(lean["status"].value_counts())

assert len(lean) == 12
assert (lean["status"] == "completed").all()

,seed,iteration_budget,repetition,actual_relative_squared_frobenius_error,status
0,0,200,0,0.780876,completed
2,0,400,0,0.792956,completed
1,0,200,1,0.780876,completed
3,0,400,1,0.792956,completed
4,1,200,0,0.780726,completed
6,1,400,0,0.813593,completed
5,1,200,1,0.780726,completed
7,1,400,1,0.813593,completed
8,2,200,0,0.780938,completed
10,2,400,0,0.792912,completed


Rows: 12
Status counts:
status
completed    12
Name: count, dtype: int64


In [20]:
import pandas as pd
from pathlib import Path

path = Path(
    "results/camvid_vgg_cp/rank512_stability/"
    "rank512_stability_summary.csv"
)

assert path.exists(), f"Missing: {path}"

df = pd.read_csv(path)

for column in [
    "rank",
    "seed",
    "iteration_budget",
    "repetition",
]:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce",
    )

lean = df[
    (df["optimization_precision"] == "float32")
    & (df["iteration_budget"].isin([200, 400]))
    & (df["repetition"].isin([0, 1]))
].copy()

print("Lean diagnostic rows:", len(lean))
print("\nStatus counts:")
print(lean["status"].value_counts(dropna=False))

display(
    lean[
        [
            "seed",
            "iteration_budget",
            "repetition",
            "actual_relative_squared_frobenius_error",
            "status",
        ]
    ].sort_values(
        ["seed", "repetition", "iteration_budget"]
    )
)

assert len(lean) == 12
assert (lean["status"] == "completed").all()

keys = lean[
    [
        "seed",
        "iteration_budget",
        "repetition",
        "optimization_precision",
    ]
].drop_duplicates()

assert len(keys) == 12

print("\nLean rank-512 diagnostic validation: PASS")

Lean diagnostic rows: 12

Status counts:
status
completed    12
Name: count, dtype: int64


,seed,iteration_budget,repetition,actual_relative_squared_frobenius_error,status
0,0,200,0,0.780876,completed
2,0,400,0,0.792956,completed
1,0,200,1,0.780876,completed
3,0,400,1,0.792956,completed
4,1,200,0,0.780726,completed
6,1,400,0,0.813593,completed
5,1,200,1,0.780726,completed
7,1,400,1,0.813593,completed
8,2,200,0,0.780938,completed
10,2,400,0,0.792912,completed



Lean rank-512 diagnostic validation: PASS


In [21]:
wide = lean.pivot(
    index=["seed", "repetition"],
    columns="iteration_budget",
    values="actual_relative_squared_frobenius_error",
).reset_index()

wide["change_200_to_400"] = (
    wide[400] - wide[200]
)

display(wide.sort_values(["seed", "repetition"]))

print("\nMean change by seed:")
display(
    wide.groupby("seed")["change_200_to_400"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

print("\nOverall change:")
print(
    wide["change_200_to_400"]
    .agg(["mean", "std", "min", "max"])
)

iteration_budget,seed,repetition,200,400,change_200_to_400
0,0,0,0.780876,0.792956,0.012080
1,0,1,0.780876,0.792956,0.012080
2,1,0,0.780726,0.813593,0.032867
3,1,1,0.780726,0.813593,0.032867
4,2,0,0.780938,0.792912,0.011974
5,2,1,0.780938,0.792912,0.011974



Mean change by seed:


,seed,mean,std,min,max
0,0,0.012080,0.0,0.012080,0.012080
1,1,0.032867,0.0,0.032867,0.032867
2,2,0.011974,0.0,0.011974,0.011974



Overall change:
mean    0.018974
std     0.010762
min     0.011974
max     0.032867
Name: change_200_to_400, dtype: float64


In [ ]:
!cd /Users/sepehrakbari/Projects/smallnet

!rsync -av \
  '/Users/sepehrakbari/Library/CloudStorage/GoogleDrive-isepehrakbari@gmail.com/My Drive/smallnet_colab_backup/camvid_vgg_cp' \
  results/camvid_vgg_cp/

!rsync -av \
  '/Users/sepehrakbari/Library/CloudStorage/GoogleDrive-isepehrakbari@gmail.com/My Drive/smallnet_colab_backup/paper' \
  results/paper/

/bin/bash: line 1: cd: /Users/sepehrakbari/Projects/smallnet: No such file or directory
sending incremental file list
rsync: [sender] change_dir "/root/Library/CloudStorage/GoogleDrive-*/My Drive/smallnet_colab_backup/camvid_vgg_cp" failed: No such file or directory (2)

sent 19 bytes  received 12 bytes  62.00 bytes/sec
total size is 0  speedup is 0.00
rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1347) [sender=3.2.7]
sending incremental file list
rsync: [sender] change_dir "/root/Library/CloudStorage/GoogleDrive-*/My Drive/smallnet_colab_backup/paper" failed: No such file or directory (2)

sent 19 bytes  received 12 bytes  62.00 bytes/sec
total size is 0  speedup is 0.00
rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1347) [sender=3.2.7]
